# E3 · Completar E2 a diez semillas: reparación de metadatos

Este cuaderno corrige el fallo `Artifact metadata must contain only finite JSON values` del primer ajuste FCN-1D, semilla 271807. El adaptador convierte la versión de Torch a texto de tipo incorporado y verifica los metadatos con el escritor original antes de comenzar un ajuste.

La recuperación conserva el intento fallido y sus **3.62394042599999 segundos** en el registro existente. Continúa las mismas **52 coordenadas de modelo y semilla**, con el límite total de **7200 segundos**. La CNN-LSTM con atención ya completó sus diez semillas y se reutilizan sus resultados.

1. Mantén la **GPU A100** y habilita el secreto **GITHUB_TOKEN** para el repositorio `jassan-bit/DeepLearning_202630`.
2. Ejecuta la celda siguiente. Puedes pegarla debajo de la celda que falló, en la misma sesión; conserva el entorno ya preparado.
3. La celda verifica la reparación publicada, archiva la carpeta local del intento fallido y continúa. Rechaza la recuperación si encuentra artefactos de entrenamiento no contemplados.

El progreso se guarda en GitHub, rama `e2-extension-10-semillas-v1`, sin Drive. Todos los ajustes y calibradores deben quedar confirmados antes de generar las nuevas predicciones de prueba. La comparación sigue siendo descriptiva y posterior al conocimiento del calendario de prueba; esta reparación no cambia las recetas ni realiza pruebas de hipótesis.


In [ ]:
# Reparación del fallo de metadatos: conserva el presupuesto y las semillas.
from pathlib import Path
from urllib.request import Request, urlopen
import hashlib
import importlib.util
import os
import sys
import time
import uuid

sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

COMMIT_REPARACION_E2 = "ede6286e344c92da6c17654de385520cf0982f0d"
ARCHIVOS_REPARACION_E2 = {
    "bootstrap.py": "3b349acf0a312da990bc2a221416c7e4e2a7a274d304b13f21130783ba479b09",
    "worker.py": "6d4f7067818c8760646dad7390d8a29e13b54c72b3c9cb7c58ff80596d030a5e",
    "prepare.py": "cb5b03de135797a1aa6cb39dbfd33743c3353455aae0cf41f027479bd542929b",
    "aggregate.py": "15cd5107eaa57ecca0b9bf842bf22b2c742ac6c3bf4004720ff6a2f0dc13aac4",
    "plan.json": "45fb6a14137b03642737521731c4b56a1312980bfbeca0319182a9ffd600cc07",
    "repair.py": "26f6607d0695546017268332c1668c98efc91f2b2952d1226ef6c8f48ac4c1f5"
}
CARPETA_REPARACION_E2 = Path("/content/e2_extension_10_semillas_v1/launcher_json_fix")
CARPETA_REPARACION_E2.mkdir(parents=True, exist_ok=True)

def descargar_reparacion_e2(nombre, esperado):
    destino = CARPETA_REPARACION_E2 / nombre
    if destino.is_symlink():
        raise RuntimeError("La ruta del código debe ser un archivo regular.")
    if destino.exists():
        if not destino.is_file() or hashlib.sha256(destino.read_bytes()).hexdigest() != esperado:
            raise RuntimeError(f"El archivo local {nombre} difiere de la reparación fijada.")
        return
    url = (
        "https://raw.githubusercontent.com/jassan-bit/DeepLearning_202630/"
        f"{COMMIT_REPARACION_E2}/experiments/e2_extension_10_semillas_v1/{nombre}"
    )
    for intento in range(3):
        try:
            solicitud = Request(url, headers={"User-Agent": "E2-metadata-repair-Colab"})
            with urlopen(solicitud, timeout=60) as respuesta:
                contenido = respuesta.read(2 * 1024 * 1024 + 1)
            if len(contenido) > 2 * 1024 * 1024 or hashlib.sha256(contenido).hexdigest() != esperado:
                raise RuntimeError(f"SHA256 incorrecto para {nombre}.")
            temporal = destino.with_name(destino.name + "." + uuid.uuid4().hex + ".tmp")
            with temporal.open("xb") as salida:
                salida.write(contenido)
            temporal.replace(destino)
            return
        except (TimeoutError, ConnectionError, OSError):
            if intento == 2:
                raise RuntimeError(f"No se pudo descargar {nombre}; el cálculo se conserva.") from None
            time.sleep(intento + 1)

for nombre_e2, sha256_e2 in ARCHIVOS_REPARACION_E2.items():
    descargar_reparacion_e2(nombre_e2, sha256_e2)

print("Reparación verificada:", COMMIT_REPARACION_E2)
print("Se conserva el consumo registrado y se continúa con los mismos 52 ajustes pendientes.")
especificacion_e2 = importlib.util.spec_from_file_location(
    "_e2_reparacion_" + uuid.uuid4().hex, CARPETA_REPARACION_E2 / "repair.py"
)
modulo_reparacion_e2 = importlib.util.module_from_spec(especificacion_e2)
especificacion_e2.loader.exec_module(modulo_reparacion_e2)
RESULTADO_EXTENSION_E2 = modulo_reparacion_e2.run()
